In [ ]:
import pandas as pd
import numpy as np
import ast
import re
from html import unescape
from sklearn.model_selection import train_test_split


# Load data
FILE_PATH = "booking_reviews copy.csv"
df = pd.read_csv(FILE_PATH)

print("Original shape:", df.shape)

# Helper functions
def safe_literal_eval(x):
    if pd.isna(x):
        return {}
    try:
        return ast.literal_eval(x)
    except Exception:
        return {}

def clean_text_basic(text):
    """
    Clean text for ML/NLP:
    - lowercase
    - remove HTML tags
    - unescape HTML entities
    - remove URLs
    - keep letters/numbers/basic punctuation
    - normalize whitespace
    """
    if pd.isna(text):
        return np.nan

    text = str(text)
    text = unescape(text)

    # remove html tags
    text = re.sub(r"<.*?>", " ", text)

    # remove urls
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # lowercase
    text = text.lower()

    text = re.sub(r"[^a-z0-9\s.,!?'-]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

def clean_text_stronger(text):
    """
    Removes most punctuation.
    """
    if pd.isna(text):
        return np.nan

    text = str(text).lower()
    text = unescape(text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

def split_tags(tag_string):
    if pd.isna(tag_string):
        return []
    return [t.strip() for t in str(tag_string).split("~") if t.strip()]

def extract_stay_nights(tags_list):
    for tag in tags_list:
        match = re.search(r"stayed\s+(\d+)\s+night", tag, flags=re.IGNORECASE)
        if match:
            return int(match.group(1))
    return np.nan

def extract_room_type(tags_list):
    """
    Try to capture room type from tags like:
    'Stayed 2 nights in Deluxe Double Room'
    """
    for tag in tags_list:
        match = re.search(
            r"stayed\s+\d+\s+night[s]?\s+in\s+(.+)",
            tag,
            flags=re.IGNORECASE
        )
        if match:
            return match.group(1).strip().lower()
    return np.nan

def extract_travel_group(tags_list):
    """
    Common Booking tags often include things like:
    - Family with young children
    - Solo traveller
    - Couple
    - Group
    """
    possible_groups = [
        "family with young children",
        "family with older children",
        "solo traveller",
        "solo traveler",
        "young couple",
        "mature couple",
        "couple",
        "group",
        "friends",
        "business traveller",
        "business traveler"
    ]

    lowered = [t.lower() for t in tags_list]
    for tag in lowered:
        for grp in possible_groups:
            if grp in tag:
                return grp
    return np.nan

def make_binary_sentiment(rating, pos_threshold=8.0):
    """
    Binary label example:
    positive = rating >= 8
    """
    if pd.isna(rating):
        return np.nan
    return int(rating >= pos_threshold)


# 3. Standardize column names
df.columns = (
    df.columns.str.strip()
              .str.lower()
              .str.replace(" ", "_")
)

# Remove duplicated rows
df = df.drop_duplicates()

if "index" in df.columns:
    df = df.drop(columns=["index"])

# Parse dates and numerics
if "reviewed_at" in df.columns:
    df["reviewed_at"] = pd.to_datetime(df["reviewed_at"], format="%d %B %Y", errors="coerce")

if "crawled_at" in df.columns:
    df["crawled_at"] = pd.to_datetime(df["crawled_at"], format="%m/%d/%Y, %H:%M:%S", errors="coerce")

for col in ["rating", "avg_rating"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


#  Parse structured columns
if "meta" in df.columns:
    df["meta_dict"] = df["meta"].apply(safe_literal_eval)
    df["review_language"] = df["meta_dict"].apply(lambda x: x.get("language"))
    df["review_source"] = df["meta_dict"].apply(lambda x: x.get("source"))

if "tags" in df.columns:
    df["tags_list"] = df["tags"].apply(split_tags)
    df["stay_nights"] = df["tags_list"].apply(extract_stay_nights)
    df["room_type"] = df["tags_list"].apply(extract_room_type)
    df["travel_group"] = df["tags_list"].apply(extract_travel_group)
else:
    df["tags_list"] = [[] for _ in range(len(df))]
    df["stay_nights"] = np.nan
    df["room_type"] = np.nan
    df["travel_group"] = np.nan


# Choose and clean the main text field
df["text_raw"] = df["review_text"]

if "raw_review_text" in df.columns:
    df["text_raw"] = df["text_raw"].fillna(df["raw_review_text"])

# Remove rows without usable text or target
df = df[~df["text_raw"].isna()].copy()
df = df[~df["rating"].isna()].copy()

# - text_clean_basic: for transformer models / modern NLP
# - text_clean_tfidf: for TF-IDF / logistic regression / linear SVM
df["text_clean_basic"] = df["text_raw"].apply(clean_text_basic)
df["text_clean_tfidf"] = df["text_raw"].apply(clean_text_stronger)

# Optional: combine title + review text for stronger signal
df["review_title"] = df["review_title"].fillna("")
df["text_combined"] = (
    df["review_title"].astype(str).str.strip() + ". " +
    df["text_clean_basic"].fillna("")
).str.strip()

# Remove very short reviews
df["text_len_chars"] = df["text_combined"].fillna("").str.len()
df["text_len_words"] = df["text_combined"].fillna("").str.split().str.len()

df = df[df["text_len_words"] >= 3].copy()


# Feature engineering
# Date features
if "reviewed_at" in df.columns:
    df["review_year"] = df["reviewed_at"].dt.year
    df["review_month"] = df["reviewed_at"].dt.month

# Missing-value handling for categorical columns
categorical_fill_cols = ["nationality", "hotel_name", "room_type", "travel_group", "review_language"]
for col in categorical_fill_cols:
    if col in df.columns:
        df[col] = df[col].fillna("unknown")

# Simple numeric fill
df["stay_nights"] = df["stay_nights"].fillna(df["stay_nights"].median())

# Derived target options
df["is_positive"] = df["rating"].apply(make_binary_sentiment)

df["rating_class"] = pd.cut(
    df["rating"],
    bins=[0, 4, 7, 10],
    labels=["low", "medium", "high"],
    include_lowest=True
)


# Leakage control
LEAKY_OR_ID_COLS = [
    "url",
    "hotel_url",
    "review_source",
]

df = df.drop(columns=[c for c in LEAKY_OR_ID_COLS if c in df.columns], errors="ignore")


# Final ML-ready dataset
ml_df = df[
    [
        "text_combined",
        "text_clean_basic",
        "text_clean_tfidf",
        "rating",
        "is_positive",
        "rating_class",
        "avg_rating",
        "hotel_name",
        "stay_nights",
        "room_type",
        "travel_group",
        "review_language",
        "review_year",
        "review_month",
        "text_len_chars",
        "text_len_words",
    ]
].copy()

print("ML-ready shape:", ml_df.shape)
print("\nMissing values:")
print(ml_df.isna().sum().sort_values(ascending=False))


# Example train/test split
train_df, test_df = train_test_split(
    ml_df,
    test_size=0.2,
    random_state=42,
    stratify=ml_df["is_positive"]
)

print("\nTrain shape:", train_df.shape)
print("Test shape:", test_df.shape)

ml_df.to_csv("booking_reviews_ml_ready.csv", index=False)
train_df.to_csv("booking_reviews_train.csv", index=False)
test_df.to_csv("booking_reviews_test.csv", index=False)

print("\nSaved:")
print("- booking_reviews_ml_ready.csv")
print("- booking_reviews_train.csv")
print("- booking_reviews_test.csv")

Original shape: (26675, 16)
ML-ready shape: (26263, 17)

Missing values:
text_combined       0
stay_nights         0
text_len_chars      0
review_month        0
review_year         0
review_language     0
travel_group        0
room_type           0
nationality         0
text_clean_basic    0
hotel_name          0
avg_rating          0
rating_class        0
is_positive         0
rating              0
text_clean_tfidf    0
text_len_words      0
dtype: int64

Train shape: (21010, 17)
Test shape: (5253, 17)

Saved:
- booking_reviews_ml_ready.csv
- booking_reviews_train.csv
- booking_reviews_test.csv
